In [0]:
from common_utils.bronze import read_raw,bronze_ingestor
from common_utils.logging import get_logger
logger=get_logger('ds2b')

In [0]:
source={
    's3sales':{
        'raw_folder':'sales',
        'file_format':'parquet',
        'read_options':{},
        'target_table':'retaildataplatform.bronze.s3sales_customers',
        'mode':'overwrite'
    },
    'cosmosdb_sales_orders':{
        'raw_folder':'cosmosdb_sales_orders',
        'file_format':'json',
        'read_options':{},
        'target_table':'retaildataplatform.bronze.cosmosdb_customers',
        'mode':'overwrite'
    },
    'sqlserver_customer':{
        'raw_folder':'sqlserver_customer',
        'file_format':'csv',
        'read_options':{'header':'true','inferSchema':'true'},
        'target_table':'retaildataplatform.bronze.sqlserver_customers',
        'mode':'overwrite'
    }
}
run_date='2026-09-19'
for source_name,config in source.items():
    raw_path=f'/Volumes/retaildataplatform/bronze/raw_data/{config['raw_folder']}/load_date={run_date}'
    logger.info('%s reading from path %s',source_name,raw_path)
    df=read_raw(spark,raw_path,file_format=config['file_format'],options=config['read_options'])
    rows=bronze_ingestor(df,mode=config['mode'],target_table=config['target_table'])
    logger.info('%s ingested %s rows',source_name,rows)
logger.info('Bronzed finished for all sources.')


In [0]:
source

In [0]:
# spark.sql("ALTER TABLE retaildataplatform.bronze.cosmosdb_customers RENAME COLUMN last_update_ts TO last_updated_ts")
raw_path='/Volumes/retaildataplatform/bronze/raw_data/cosmosdb_sales_orders/load_date=2026-09-19/'
target_table='retaildataplatform.bronze.cosmosdb_customers'
df_cosmos=read_raw(spark,raw_path,file_format='json',options=None)

bronze_ingestor(df_cosmos, mode='overwrite', target_table=target_table)

In [0]:
raw_path= "/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customer/load_date=2026-09-19/"
target_table= "retaildataplatform.bronze.sqlserver_customers"
options={'header':'true','inferSchema':'true'}
df_sqlserver=read_raw(spark, raw_path, file_format='csv', options = options)

bronze_ingestor(df_sqlserver, mode='overwrite', target_table=target_table)

In [0]:
raw_path='/Volumes/retaildataplatform/bronze/raw_data/sales/load_date=2026-09-19/'
target_path='retaildataplatform.bronze.s3Sales_customers'
options={}
df_s3sales=read_raw(spark,raw_path,file_format='parquet',options=options)

bronze_ingestor(df_s3sales,mode='overwrite',target_table=target_path)